# Workforce Data Validation

This notebook runs Story 02 schema validation and basic quality profiling for the four raw workforce CSV files.

In [1]:
from pathlib import Path
import sys

import polars as pl
import yaml
from IPython.display import Markdown, display

workspace_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists() and (path / 'artifacts').exists())
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from shared.src.data_processing.workforce_validation import validate_workforce_files

raw_dir = workspace_root / 'shared' / 'data' / '1_raw' / 'workforce'
file_paths = [
    raw_dir / 'doctors.csv',
    raw_dir / 'nurses.csv',
    raw_dir / 'pharmacists.csv',
    raw_dir / 'physiotherapists.csv',
]
file_paths

[PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/doctors.csv'),
 PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/nurses.csv'),
 PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/pharmacists.csv'),
 PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/physiotherapists.csv')]

In [2]:
results = validate_workforce_files(file_paths)
results

2026-04-22 23:10:49.805 | INFO     | shared.src.data_processing.workforce_validation:validate_workforce_file:55 - Validating workforce file: /Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/doctors.csv
2026-04-22 23:10:49.809 | INFO     | shared.src.data_processing.workforce_validation:validate_workforce_file:126 - doctors.csv schema=pass duplicates=0 negative_count_rows=0 year_range=2006..2019
2026-04-22 23:10:49.810 | INFO     | shared.src.data_processing.workforce_validation:validate_workforce_file:55 - Validating workforce file: /Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/nurses.csv
2026-04-22 23:10:49.811 | INFO     | shared.src.data_processing.workforce_validation:validate_workforce_file:126 - nurses.csv schema=pass duplicates=0 negative_count_rows=0 year_range=2006..2019
2026-04-22 23:10:49.811 | INFO     | shared.src.data_processing.workforce_validation:validate_workforce_file:55 - Vali

{'doctors': {'file_name': 'doctors.csv', 'row_count': 78, 'schema_validation': 'pass', 'null_rates': {'year': 0.0, 'sector': 0.0, 'count': 0.0}, 'duplicate_row_count': 0, 'negative_count_rows': 0, 'year_range': {'min': 2006, 'max': 2019}, 'unique_sector_values': ['not in active practice', 'private', 'public']},
 'nurses': {'file_name': 'nurses.csv', 'row_count': 126, 'schema_validation': 'pass', 'null_rates': {'year': 0.0, 'sector': 0.0, 'count': 0.0}, 'duplicate_row_count': 0, 'negative_count_rows': 0, 'year_range': {'min': 2006, 'max': 2019}, 'unique_sector_values': ['not in active practice', 'private', 'public']},
 'pharmacists': {'file_name': 'pharmacists.csv', 'row_count': 42, 'schema_validation': 'pass', 'null_rates': {'year': 0.0, 'sector': 0.0, 'count': 0.0}, 'duplicate_row_count': 0, 'negative_count_rows': 0, 'year_range': {'min': 2006, 'max': 2019}, 'unique_sector_values': ['not in active practice', 'private', 'public']},
 'physiotherapists': {'file_name': 'physiotherapists.c

In [3]:
summary_rows = []
for profession, finding in results.items():
    summary_rows.append({
        'file': finding['file_name'],
        'schema_validation': finding['schema_validation'],
        'rows': finding['row_count'],
        'duplicates': finding.get('duplicate_row_count', 0),
        'negative_count_rows': finding.get('negative_count_rows', 0),
        'year_min': finding.get('year_range', {}).get('min'),
        'year_max': finding.get('year_range', {}).get('max'),
        'sectors': ', '.join(finding.get('unique_sector_values', [])),
    })

summary_df = pl.DataFrame(summary_rows)
summary_df

shape: (4, 8)
┌──────────────────────┬───────────────────┬──────┬────────────┬─────────────────────┬──────────┬──────────┬─────────────────────────────────────────┐
│ file                 ┆ schema_validation ┆ rows ┆ duplicates ┆ negative_count_rows ┆ year_min ┆ year_max ┆ sectors                                 │
│ ---                  ┆ ---               ┆ ---  ┆ ---        ┆ ---                 ┆ ---      ┆ ---      ┆ ---                                     │
│ str                  ┆ str               ┆ i64  ┆ i64        ┆ i64                 ┆ i64      ┆ i64      ┆ str                                     │
╞══════════════════════╪═══════════════════╪══════╪════════════╪═════════════════════╪══════════╪══════════╪═════════════════════════════════════════╡
│ doctors.csv          ┆ pass              ┆ 78   ┆ 0          ┆ 0                   ┆ 2006     ┆ 2019     ┆ not in active practice, private, publi… │
│ nurses.csv           ┆ pass              ┆ 126  ┆ 0          ┆ 0              

In [4]:
display(Markdown('## YAML Preview'))
yaml.safe_dump(results, sort_keys=False)

## YAML Preview

'doctors:
  file_name: doctors.csv
  row_count: 78
  schema_validation: pass
  null_rates:
    year: 0.0
    sector: 0.0
    count: 0.0
  duplicate_row_count: 0
  negative_count_rows: 0
  year_range:
    min: 2006
    max: 2019
  unique_sector_values:
  - not in active practice
  - private
  - public
...
'